# 06 | Hugging Face学習済みAI × Genesis 3D物理巡回

**実行順：**00（描画動作確認）→04（学習、`road_ml_assets.zip`生成）→06（このNotebook）。05のPPOは任意。

**物理接続**：AIが分類した仮想道路カメラ画像 → ルール定義リスク → 点検先選択と速度上限制御 → Genesis車輪関節への速度指令 → 3D剛体位置のフィードバック。

**誤認防止**：外部画像はHugging Faceの写真であり、**Genesisカメラが3D世界を撮影したピクセルではありません**。Genesisの3Dカメラはロボット動作を可視化する用途です。実際の街路画像をGenesisレンダラから取得したと主張しないでください。

**警告**：Colab GPU/EGLでのGenesis実行はこの環境では未検証。描画動作を必ずNotebook 00で確認してください。

In [1]:
!apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libgl1-mesa-dri libegl-mesa0 libosmesa6 ffmpeg >/dev/null
%pip -q install 'genesis-world==1.4.4' 'numpy<3' 'scipy<2' 'scikit-learn>=1.3,<2' 'pandas>=2,<3' 'opencv-python-headless>=4.8,<5' 'stable-baselines3>=2.3,<3' imageio-ffmpeg pillow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
%%writefile pai_patrol.py
"""Physical AI: risk-aware mobile inspection robot (Colab, reproducible demo).

Simulation fidelity: 2-D occupancy grid + differential-drive/unicycle kinematics,
not a ROS/Gazebo or hardware experiment. Remote sensors are synthetic, not measured.
Anomaly ground truth is private to the environment and used only for scoring.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import heapq
import math
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

STATIONS = {
    'A': (1.5, 7.4),
    'B': (3.5, 6.1),
    'C': (8.6, 7.0),
    'D': (10.5, 3.4),
    'E': (6.5, 1.3),
    'F': (2.0, 3.6),
}
BASE = (1.0, 1.0)
OBSTACLES = [(3.9, 2.1, 5.0, 4.8), (6.5, 4.1, 8.0, 5.7), (8.1, 1.5, 9.0, 3.2)]
WIDTH, HEIGHT = 12.0, 9.0
FEATURES = ['temperature_C', 'vibration_mm_s', 'current_A', 'sound_dB']
STATION_KEYS = list(STATIONS)
MEANS = np.array([[35 + i*1.2, 2.1+i*.08, 5+i*.13, 51+i*.45] for i in range(len(STATIONS))], dtype = float)
STDS = np.array([1.6, .35, .26, 2.1])
FAULT_SHIFT = np.array([9.0, 1.8, 1.2, 9.0])

@dataclass(frozen = True)
class Config:
    dt: float = 0.10
    horizon: float = 150.0
    max_speed: float = 0.68
    max_turn: float = 2.4
    robot_radius: float = 0.25
    grid_res: float = 0.25
    remote_period: float = 2.0
    camera_fps: int = 10
    camera_stride: int = 1
    risk_gate: float = 1.10
    revisit_after: float = 15.0

CFG = Config()

class FloorMap:
    def __init__(self, cfg = CFG):
        self.cfg = cfg
        self.nx = int(round(WIDTH/cfg.grid_res))
        self.ny = int(round(HEIGHT/cfg.grid_res))
        self.grid = np.zeros((self.ny, self.nx), dtype = np.uint8)
        for gy in range(self.ny):
            for gx in range(self.nx):
                x, y = self.from_cell((gx, gy))
                if self.blocked(x, y, margin = cfg.robot_radius+0.09):
                    self.grid[gy, gx] = 1
        assert all(self.grid[self.to_cell(pt)[1], self.to_cell(pt)[0]] == 0 for pt in [BASE, *STATIONS.values()]), 'Station collides with inflated obstacles'
    def blocked(self, x, y, margin = 0.):
        if x<margin or y<margin or x>WIDTH-margin or y>HEIGHT-margin:
            return True
        return any(a-margin <= x <= c+margin and b-margin <= y <= d+margin for a, b, c, d in OBSTACLES)
    def to_cell(self, xy):
        return (min(self.nx-1,max(0,int(xy[0]/self.cfg.grid_res))),
                min(self.ny-1, max(0, int(xy[1]/self.cfg.grid_res))))
    def from_cell(self, cell):
        return ((cell[0]+.5)*self.cfg.grid_res, (cell[1]+.5)*self.cfg.grid_res)
    def line_clear(self, a, b):
        d = math.dist(a, b)
        for u in np.linspace(0, 1, max(2, int(math.ceil(d/0.09)))):
            x = a[0]+(b[0]-a[0])*u
            y = a[1]+(b[1]-a[1])*u
            if self.blocked(x, y, self.cfg.robot_radius+0.04):
                return False
        return True
    def plan(self, start, end):
        s = self.to_cell(start)
        g = self.to_cell(end)
        if s == g or self.line_clear(start, end):
            return [tuple(end)]
        h = lambda p: math.hypot(p[0]-g[0], p[1]-g[1])
        heap = [(h(s), 0., s)]
        best = {s:0.}
        parent = {}
        found = False
        dirs = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1) if dx or dy]
        while heap:
            _, cost, u = heapq.heappop(heap)
            if cost>best.get(u, float('inf'))+1e-8:
                continue
            if u == g:
                found = True
                break
            for dx, dy in dirs:
                v = (u[0]+dx, u[1]+dy)
                if not (0 <= v[0]<self.nx and 0 <= v[1]<self.ny) or self.grid[v[1], v[0]]:
                    continue
                if dx and dy and (self.grid[u[1], v[0]] or self.grid[v[1], u[0]]):
                    continue
                nc = cost+math.hypot(dx, dy)
                if nc<best.get(v, float('inf')):
                    best[v] = nc
                    parent[v] = u
                    heapq.heappush(heap, (nc+h(v), nc, v))
        if not found:
            raise RuntimeError(f'Route unreachable: {start} -> {end}')
        path = [g]
        while path[-1] != s:
            path.append(parent[path[-1]])
        path.reverse()
        pts = [tuple(start)]+[self.from_cell(pt) for pt in path[1:-1]]+[tuple(end)]
        # Greedy line-of-sight waypoint reduction (collision checked, inflated obstacles)
        out = []
        i = 0
        while i<len(pts)-1:
            j = len(pts)-1
            while j>i+1 and not self.line_clear(pts[i], pts[j]):
                j -= 1
            out.append(pts[j])
            i = j
        return out
    def distance(self, start, end):
        p = [tuple(start)]+self.plan(start, end)
        return sum(math.dist(p[i], p[i+1]) for i in range(len(p)-1))

@dataclass(frozen = True)
class Fault:
    station: str
    onset: float
    severity: float

@dataclass(frozen = True)
class Scenario:
    seed: int
    faults: tuple[Fault, ...]


def make_scenario(seed):
    rng = np.random.default_rng(int(seed))
    keys = rng.choice(STATION_KEYS, size = 2, replace = False)
    return Scenario(int(seed),(Fault(str(keys[0]),0.0,float(rng.uniform(.95,1.45))),
                               Fault(str(keys[1]), float(rng.choice([12., 18., 24.])), float(rng.uniform(1.05, 1.55)))))


def telemetry(scenario, station, t, mode = 'remote', replicate = 0):
    """Synthetic sensor. Policy receives sensor values only, never `scenario.faults`."""
    idx = STATION_KEYS.index(station)
    # Deterministic per (seed, equipment, time-window, mode, replicate).
    tick = max(0, int(t//(CFG.remote_period if mode == 'remote' else 1.0)))
    ss = np.random.SeedSequence([scenario.seed, idx, tick, 0 if mode == 'remote' else 1, int(replicate)])
    rng = np.random.default_rng(ss)
    noise = 1.17 if mode == 'remote' else .40
    x = MEANS[idx]+rng.normal(size = 4)*STDS*noise
    for fault in scenario.faults:
        if station == fault.station and t >= fault.onset:
            x = x+FAULT_SHIFT*fault.severity
    return x

class Detector:
    def __init__(self, seed = 2026):
        rng = np.random.default_rng(seed)
        normal = rng.normal(size = (3500, 4))*1.17
        # Only NORMAL observations in training. Ground truth of evaluation faults unseen.
        self.model = IsolationForest(n_estimators = 120, max_samples = 256, random_state = seed, n_jobs = -1)
        self.model.fit(normal)
        vr = rng.normal(size = (3000, 4))*1.17
        vl = rng.normal(size = (3000, 4))*.40
        rs = -self.model.score_samples(vr)
        ls = -self.model.score_samples(vl)
        self.remote_threshold = float(np.quantile(rs, .99))
        self.local_threshold = float(np.quantile(ls, .995))
        self.reference_75 = float(np.quantile(rs, .75))
        self.spread = max(.015, self.remote_threshold-self.reference_75)
    def features(self, x, station):
        idx = STATION_KEYS.index(station)
        return ((np.asarray(x)-MEANS[idx])/STDS).reshape(1, -1)
    def anomaly(self, x, station):
        return float(-self.model.score_samples(self.features(x, station))[0])
    def remote_risk(self, x, station):
        a = self.anomaly(x, station)
        return float(np.clip((a-self.reference_75)/self.spread, 0., 3.0))
    def onsite_flag(self, scenario, station, t):
        # independent repeat-measurement median reduces occasional false alarms
        scores = [self.anomaly(telemetry(scenario, station, t, mode = 'local', replicate = k), station) for k in range(3)]
        med = float(np.median(scores))
        return med>self.local_threshold, med

@dataclass
class PatrolResult:
    policy: str
    scenario: Scenario
    metrics: dict
    visits: list
    trajectory: list
    events: list
    remote_history: list

class Patrol:
    def __init__(self, scenario, detector, policy = 'adaptive', cfg = CFG, floor = None):
        if policy not in ('adaptive','nearest'):
            raise ValueError('policy must be adaptive or nearest')
        self.scenario = scenario
        self.detector = detector
        self.policy = policy
        self.cfg = cfg
        self.floor = floor if floor is not None else FloorMap(cfg)
        self.pos = np.array(BASE, dtype = float)
        self.heading = math.pi/2
        self.t = 0.
        self.distance = 0.
        self.collisions = 0
        self.replans = 0
        self.route = []
        self.target = None
        self.waypoint = 0
        self.visits = []
        self.events = []
        self.history = []
        self.trajectory = []
        self.last_visit = {k:-1e9 for k in STATION_KEYS}
        self.visit_counts = {k:0 for k in STATION_KEYS}
        self.target_start = -1e9
        self.consecutive_alarm = {k:0 for k in STATION_KEYS}
        self.remote_risks = {k:0. for k in STATION_KEYS}
        self.active_alarm = {k:False for k in STATION_KEYS}
        self.detections = {}
        self.flagged_stations = set()
        self.fp_count = 0
        self.next_remote = 0.
        self.finished = False
    def observe_remote(self):
        risks = {}
        for key in STATION_KEYS:
            observed = telemetry(self.scenario, key, self.t,'remote')
            risk = self.detector.remote_risk(observed, key)
            self.remote_risks[key] = .35*self.remote_risks[key]+.65*risk
            self.consecutive_alarm[key] = (self.consecutive_alarm[key]+1 if self.remote_risks[key] >= self.cfg.risk_gate else 0)
            risks[key] = self.remote_risks[key]
        self.history.append({'t':round(self.t, 1), **risks})
    def options(self):
        outstanding = [k for k in STATION_KEYS if self.last_visit[k]<0]
        if self.policy == 'nearest':
            return outstanding
        alerts=[k for k in STATION_KEYS if self.consecutive_alarm[k]>=2
                and k not in self.flagged_stations and self.visit_counts[k]<2
                and (self.t-self.last_visit[k])>self.cfg.revisit_after]
        return sorted(set(outstanding+alerts))
    def choose_target(self):
        choices = self.options()
        if not choices:
            return 'HOME'
        distances = {k:self.floor.distance(self.pos, STATIONS[k]) for k in choices}
        if self.policy == 'nearest':
            return min(choices, key = lambda k:(distances[k], k))
        # Alarm priority competes with travel cost; no hidden ground truth labels used.
        return max(choices,key=lambda k:(3.5*self.remote_risks[k]-.30*distances[k]
                                         + (1.8 if self.last_visit[k]<0 else 0), -distances[k]))
    def set_target(self, k):
        if self.target == k:
            return
        self.target = k
        self.replans += 1
        self.target_start = self.t
        end = BASE if k == 'HOME' else STATIONS[k]
        self.route = self.floor.plan(self.pos, end)
        self.waypoint = 0
        self.events.append({'t':round(self.t, 1),'kind':'reroute','station':k})
    def arrive(self):
        k = self.target
        if k == 'HOME':
            self.finished = True
            self.events.append({'t':round(self.t, 1),'kind':'home','station':'HOME'})
            return
        flagged, score = self.detector.onsite_flag(self.scenario, k, self.t)
        fault = next((f for f in self.scenario.faults if f.station == k and self.t >= f.onset), None)
        is_fault = fault is not None
        if flagged:
            self.flagged_stations.add(k)
        if flagged and is_fault:
            self.detections.setdefault(k, self.t)
        if flagged and not is_fault:
            self.fp_count += 1
        self.last_visit[k] = self.t
        self.visit_counts[k] += 1
        row={'t':round(self.t,1),'station':k,'flagged':bool(flagged),'fault_active':bool(is_fault),
             'anomaly_score':score,'risk':self.remote_risks[k]}
        self.visits.append(row)
        self.events.append({'t':round(self.t,1),'kind':'inspect','station':k,
                            'flagged':bool(flagged),'fault_active':bool(is_fault)})
        self.target = None
        self.route = []
    def step(self):
        if self.finished:
            return
        if self.t+1e-8 >= self.next_remote:
            self.observe_remote()
            self.next_remote += self.cfg.remote_period
            if self.policy == 'adaptive' and self.target not in (None,'HOME') and self.t-self.target_start>6.:
                # A fresh strong remote alarm can interrupt a low-risk current route.
                opts = self.options()
                serious = [k for k in opts if k != self.target and self.consecutive_alarm[k] >= 2 and self.remote_risks[k]>self.remote_risks.get(self.target, 0)+1.05]
                if serious:
                    best = max(serious, key = lambda k:self.remote_risks[k]-.18*self.floor.distance(self.pos, STATIONS[k]))
                    self.set_target(best)
        if self.target is None:
            self.set_target(self.choose_target())
        goal = np.asarray(BASE if self.target == 'HOME' else STATIONS[self.target])
        if math.dist(self.pos, goal)<.37:
            self.arrive()
        else:
            if self.waypoint >= len(self.route):
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
            wp = np.asarray(self.route[self.waypoint])
            if np.linalg.norm(wp-self.pos)<.22 and self.waypoint<len(self.route)-1:
                self.waypoint += 1
                wp = np.asarray(self.route[self.waypoint])
            desired = math.atan2(wp[1]-self.pos[1], wp[0]-self.pos[0])
            delta = (desired-self.heading+math.pi)%(2*math.pi)-math.pi
            omega = np.clip(3.0*delta, -self.cfg.max_turn, self.cfg.max_turn)
            # Differential drive kinematics: v=(r/2)(wr+wl), w=(r/L)(wr-wl)
            v = self.cfg.max_speed * max(.05, 1.0-min(1.0, abs(delta)/1.3))
            # Convert commanded twist to actual left/right wheel rates, then integrate.
            wheel_radius, wheel_separation = .085, .36
            wheel_l = (v-.5*wheel_separation*omega)/wheel_radius
            wheel_r = (v+.5*wheel_separation*omega)/wheel_radius
            v_eff = .5*wheel_radius*(wheel_r+wheel_l)
            omega_eff = wheel_radius*(wheel_r-wheel_l)/wheel_separation
            next_heading = (self.heading+omega_eff*self.cfg.dt+math.pi)%(2*math.pi)-math.pi
            proposed = self.pos+v_eff*self.cfg.dt*np.array([math.cos(next_heading), math.sin(next_heading)])
            if self.floor.blocked(*proposed, margin = self.cfg.robot_radius+.01):
                self.collisions += 1
                self.route = self.floor.plan(self.pos, goal)
                self.waypoint = 0
                self.replans += 1
                v = 0.0
            else:
                self.pos = proposed
                self.distance += v*self.cfg.dt
            self.heading = next_heading
        self.t += self.cfg.dt
        self.trajectory.append((self.t,float(self.pos[0]),float(self.pos[1]),float(self.heading),self.target or '-',
                                float(max(self.remote_risks.values())), len(self.detections)))
    def run(self):
        nsteps = int(self.cfg.horizon/self.cfg.dt)
        for _ in range(nsteps):
            if self.finished:
                break
            self.step()
        fault_count = len(self.scenario.faults)
        delays = []
        for f in self.scenario.faults:
            found = self.detections.get(f.station)
            # Missed events receive the horizon penalty (censored observation).
            delays.append((self.cfg.horizon-f.onset if found is None else max(0, found-f.onset), f.severity))
        weight = sum(sev for _, sev in delays)
        weighted = sum(d*sev for d, sev in delays)/weight if weight else float('nan')
        inspected = len([k for k in STATION_KEYS if self.last_visit[k] >= 0])
        metrics={'seed':self.scenario.seed,'policy':self.policy,'detected':len(self.detections),
                 'faults':fault_count,'recall':len(self.detections)/fault_count,
                 'false_alarm_inspections':self.fp_count,
                 'weighted_detection_delay_s':weighted,'distance_m':self.distance,
                 'elapsed_s':self.t,'coverage':inspected/len(STATION_KEYS),
                 'replans':self.replans,'collision_attempts':self.collisions,
                 'home_reached':self.finished}
        return PatrolResult(self.policy, self.scenario, metrics, self.visits, self.trajectory, self.events, self.history)


def evaluate(seed_list = range(30, 50), detector = None, cfg = CFG, verbose = True):
    detector = detector if detector is not None else Detector()
    floor = FloorMap(cfg)
    results = []
    scenarios = []
    for seed in seed_list:
        s = make_scenario(seed)
        scenarios.append(s)
        for policy in ('nearest','adaptive'):
            results.append(Patrol(s, detector, policy, cfg, floor).run())
    dataframe = pd.DataFrame([r.metrics for r in results])
    paired = dataframe.pivot(index = 'seed', columns = 'policy')
    differences=pd.DataFrame({
        'delay_improvement_s':paired[('weighted_detection_delay_s','nearest')]-paired[('weighted_detection_delay_s','adaptive')],
        'recall_improvement':paired[('recall','adaptive')]-paired[('recall','nearest')],
        'extra_distance_m':paired[('distance_m','adaptive')]-paired[('distance_m','nearest')],
    })
    if verbose:
        print('Policies: nearest = distance-only patrol; adaptive = AI-risk priority patrol')
        print('Note: training/calibration and evaluation scenarios are separate.')
        print(dataframe.groupby('policy')[['recall','weighted_detection_delay_s','distance_m','coverage','collision_attempts','false_alarm_inspections']].mean().round(2).to_string())
        print('Paired mean differences (positive delay_improvement is better):')
        print(differences.mean().round(2).to_string())
    return dataframe, differences, results


def smoke_test():
    floor = FloorMap()
    d = Detector()
    for k, p in STATIONS.items():
        assert floor.plan(BASE, p), k
    scenario = make_scenario(42)
    v = Patrol(scenario, d,'adaptive', floor = floor).run()
    assert 0 <= v.metrics['recall'] <= 1
    assert len(v.trajectory)>40
    assert not np.isnan(v.metrics['weighted_detection_delay_s'])
    assert v.metrics['elapsed_s'] <= CFG.horizon+1e-6
    print('smoke test OK:', {k:round(v.metrics[k], 2) if isinstance(v.metrics[k], float) else v.metrics[k] for k in ['recall','distance_m','elapsed_s','coverage','home_reached','collision_attempts']})
    return v

if __name__ == '__main__':
    smoke_test()
    evaluate(range(42, 46))


Overwriting pai_patrol.py


In [3]:
%%writefile genesis_mission.py
"""Genesis World 3-D rigid-body mobile rover driven by wheel velocity commands.

All observations are synthetic; the simulator is a *simulated* factory. No hardware validation.
The policy receives only synthetic sensor observations, never ground-truth fault IDs/times.
3-D dynamic motion is separately logged; planner-only benchmark is a DIFFERENT experiment.
"""
from __future__ import annotations
import json
import math
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from pai_patrol import (BASE, STATIONS, OBSTACLES, STATION_KEYS, CFG,
                        Detector, Patrol, FloorMap, make_scenario, Scenario)

@dataclass
class MissionSettings:
    dt: float = 0.05
    duration_s: float = 90.0
    fps: int = 10
    backend: str = 'cuda'  # 'cpu' option for diagnostics
    res: tuple[int, int] = (640, 480)
    policy: str = 'adaptive'  # 'nearest' or 'adaptive'
    seed: int = 42
    max_v: float = 0.65
    max_omega: float = 1.9
    wheel_radius: float = 0.12
    wheel_track: float = 0.41
    # URDF positive y wheel axes rotate the rover toward -x for positive omega_y
    wheel_sign: float = -1.0


def generate_rover_urdf(path: Path):
    """Create a simple explicitly inertialized differential drive URDF, no external assets."""
    path = Path(path)
    path.parent.mkdir(parents = True, exist_ok = True)
    text = '''<?xml version="1.0"?>
<robot name="pai_differential_rover">
 <material name="blue"><color rgba="0.13 0.39 0.78 1"/></material>
 <material name="dark"><color rgba="0.12 0.15 0.20 1"/></material>
 <material name="accent"><color rgba="0.95 0.65 0.20 1"/></material>
 <link name="base_link">
   <inertial><origin xyz="0 0 0"/><mass value="6"/><inertia ixx="0.14" ixy="0" ixz="0" iyy="0.18" iyz="0" izz="0.20"/></inertial>
   <visual><geometry><box size="0.50 0.32 0.13"/></geometry><material name="blue"/></visual>
   <collision><geometry><box size="0.50 0.32 0.13"/></geometry></collision>
 </link>
 <link name="left_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="left_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="left_wheel"/><origin xyz="-0.05 0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="right_wheel">
  <inertial><origin xyz="0 0 0"/><mass value="0.5"/><inertia ixx="0.002" ixy="0" ixz="0" iyy="0.002" iyz="0" izz="0.002"/></inertial>
  <visual><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry><material name="dark"/></visual>
  <collision><origin rpy="1.57079632679 0 0"/><geometry><cylinder radius="0.12" length="0.06"/></geometry></collision>
 </link>
 <joint name="right_wheel_joint" type="continuous">
  <parent link="base_link"/><child link="right_wheel"/><origin xyz="-0.05 -0.205 -0.13" rpy="0 0 0"/>
  <axis xyz="0 1 0"/><limit effort="20" velocity="25"/><dynamics damping="0.03" friction="0.01"/>
 </joint>
 <link name="caster_ball">
  <inertial><origin xyz="0 0 0"/><mass value="0.1"/><inertia ixx="0.00015" ixy="0" ixz="0" iyy="0.00015" iyz="0" izz="0.00015"/></inertial>
  <visual><geometry><sphere radius="0.06"/></geometry><material name="accent"/></visual>
  <collision><geometry><sphere radius="0.06"/></geometry></collision>
 </link>
 <joint name="caster_joint" type="fixed"><parent link="base_link"/><child link="caster_ball"/><origin xyz="0.19 0 -0.19"/></joint>
</robot>'''
    path.write_text(text, encoding = 'utf8')
    return path


def as_numpy(t):
    return t.detach().cpu().numpy() if hasattr(t, 'detach') else np.asarray(t)


def xy_yaw(robot):
    xyz = as_numpy(robot.get_pos()).reshape(-1, 3)[0]
    w, x, y, z = as_numpy(robot.get_quat()).reshape(-1, 4)[0]
    yaw = math.atan2(2*(w*z+x*y), 1-2*(y*y+z*z))
    return (float(xyz[0]), float(xyz[1])), float(yaw), float(xyz[2])


def make_scene(settings:MissionSettings, output:Path):
    import genesis as gs
    if settings.dt <= 0 or settings.duration_s <= 0:
        raise ValueError('Invalid time settings')
    output = Path(output)
    output.mkdir(parents = True, exist_ok = True)
    scene=gs.Scene(show_viewer=False,
                   sim_options = gs.options.SimOptions(dt = settings.dt),
                   vis_options = gs.options.VisOptions(show_world_frame = False))
    scene.add_entity(gs.morphs.Plane(), surface = gs.surfaces.Plastic(color = (.80, .84, .88, 1)))
    # Ground obstacles: rectangles come from same grid as the planner.
    for i, (x1, y1, x2, y2) in enumerate(OBSTACLES):
        scene.add_entity(gs.morphs.Box(pos=((x1+x2)/2,(y1+y2)/2,.45),
                                      size = (x2-x1, y2-y1, .9), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.38,.42,.48,1)), name=f'obstacle_{i}')
    for k, (x, y) in STATIONS.items():
        # Device cabinets are deliberately offset from collision-free inspection points.
        scene.add_entity(gs.morphs.Box(pos=(x+.58,y+.50,.36),
                                      size = (.42, .34, .72), fixed = True),
                         surface=gs.surfaces.Plastic(color=(.16,.50,.45,1)), name=f'equipment_{k}')
        scene.add_entity(gs.morphs.Box(pos=(x,y,.012),size=(.22,.22,.024),fixed=True,collision=False),
                         surface=gs.surfaces.Plastic(color=(.98,.76,.12,1)), name=f'inspect_marker_{k}')
    urdf = generate_rover_urdf(output/'pai_rover.urdf')
    rover=scene.add_entity(gs.morphs.URDF(file=str(urdf.resolve()),
                                           pos = (BASE[0], BASE[1], .25),
                                           fixed = False, merge_fixed_links = True),
                           name = 'pai_rover')
    camera=scene.add_camera(res=settings.res,
                            pos = (15.8, -13.0, 17.8), lookat = (6.0, 4.2, .0), fov = 48, GUI = False)
    scene.build()
    indices = [int(rover.get_joint(n).dofs_idx_local[0]) for n in ('left_wheel_joint','right_wheel_joint')]
    rover.set_dofs_kp(np.zeros(2), indices)
    rover.set_dofs_kv(np.array([7., 7.]), indices)
    rover.set_dofs_force_range(lower = np.array([-18., -18.]), upper = np.array([18., 18.]), dofs_idx_local = indices)
    return scene, rover, camera, indices


def run_3d(settings:MissionSettings, detector:Detector, output:Path, smoke = False):
    """3D wheel-actuated closed-loop run. Returns physical metrics, separate from 2D benchmark."""
    scene, rover, camera, joints = make_scene(settings, output)
    scenario = make_scenario(settings.seed)
    p = Patrol(scenario, detector, policy = settings.policy)
    phys_rows = []
    events = []
    visits = []
    path=Path(output)/('genesis_smoke.mp4' if smoke else f'genesis_{settings.policy}.mp4')
    total_s = 2.0 if smoke else settings.duration_s
    steps = round(total_s/settings.dt)
    last_xy = None
    physical_dist = 0.
    boundaries = 0
    ground_contact_flags = 0
    # Ground truth stays in `scenario`; Patrol only uses it in evaluation and onsite checks.
    camera.start_recording(save_to_filename = str(path.resolve()), fps = settings.fps)
    try:
        for i in range(steps):
            p.t = i*settings.dt
            pos, theta, z = xy_yaw(rover)
            p.pos = np.array(pos)
            p.heading = theta
            if last_xy is not None:
                physical_dist += math.dist(pos, last_xy)
            last_xy = pos
            if p.floor.blocked(*pos, margin = .12):
                boundaries += 1 # geofence/proximity proxy, NOT a measured collision impulse
            if z<-.05:
                ground_contact_flags += 1
            if not smoke:
                if p.t + 1e-6 >= p.next_remote:
                    p.observe_remote()
                    p.next_remote += p.cfg.remote_period
                    if settings.policy == 'adaptive' and p.target not in (None,'HOME') and p.t-p.target_start>6:
                        choices = p.options()
                        serious=[k for k in choices if k!=p.target and
                                p.consecutive_alarm[k] >= 2 and
                                p.remote_risks[k]>p.remote_risks.get(p.target, 0)+1.05]
                        if serious:
                            best = max(serious, key = lambda k:p.remote_risks[k]-.18*p.floor.distance(p.pos, STATIONS[k]))
                            p.set_target(best)
                if p.target is None and not p.finished:
                    p.set_target(p.choose_target())
            else:
                if p.target is None:
                    p.set_target('A')
            v = 0.
            omega = 0.
            if p.target is not None and not p.finished:
                target = np.asarray(BASE if p.target == 'HOME' else STATIONS[p.target])
                if np.linalg.norm(target-p.pos)<.40:
                    if smoke:
                        p.finished = True
                    else:
                        p.arrive()
                if not p.finished and p.target is not None:
                    if p.waypoint >= len(p.route):
                        p.route = p.floor.plan(p.pos, target)
                        p.waypoint = 0
                        p.replans += 1
                    wp = np.asarray(p.route[p.waypoint])
                    if np.linalg.norm(wp-p.pos)<.28 and p.waypoint<len(p.route)-1:
                        p.waypoint += 1
                        wp = np.asarray(p.route[p.waypoint])
                    error = (math.atan2(wp[1]-p.pos[1], wp[0]-p.pos[0])-theta+math.pi)%(2*math.pi)-math.pi
                    omega = float(np.clip(2.2*error, -settings.max_omega, settings.max_omega))
                    v = float(settings.max_v*max(0, 1-abs(error)/.80))
            # Twist -> joint velocities. Sign is determined by URDF wheel joint axes.
            wl = settings.wheel_sign*(v-settings.wheel_track*omega/2)/settings.wheel_radius
            wr = settings.wheel_sign*(v+settings.wheel_track*omega/2)/settings.wheel_radius
            rover.control_dofs_velocity(np.asarray([wl, wr]), dofs_idx_local = joints)
            scene.step()
            phys_rows.append(dict(t=p.t,x=pos[0],y=pos[1],z=z,yaw=theta,
                                  v_command = v, omega_command = omega,
                                  target = p.target or 'NONE',
                                  max_ai_risk = max(p.remote_risks.values()),
                                  completed = p.finished))
            if p.finished and not smoke:
                # Continue recording a stationary robot to preserve a 60-180s video.
                rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        # Recording is also stopped in finally.
    finally:
        try:
            rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        except Exception:
            pass
        camera.stop_recording()
    phys = pd.DataFrame(phys_rows)
    phys.to_csv(Path(output)/'genesis_physical_trace.csv', index = False)
    pd.DataFrame(p.visits).to_csv(Path(output)/'genesis_visits.csv', index = False)
    pd.DataFrame(p.history).to_csv(Path(output)/'genesis_ai_risks.csv', index = False)
    # Note: p.events is the policy control event log; 3D distance derived from rigid body position.
    (Path(output)/'genesis_events.json').write_text(json.dumps(p.events, indent = 2, ensure_ascii = False))
    result={'policy':settings.policy,'seed':settings.seed,'simulated_3d_seconds':total_s,
            'physical_distance_m':physical_dist,'inspected_stations':len(p.visits),
            'replans':p.replans,'geofence_proximity_steps_proxy':boundaries,
            'below_floor_steps':ground_contact_flags,'video':str(path),
            'method':'Genesis wheel-joint velocity control and base pose feedback; synthetic telemetry',
            'warning':'Not a real-world robot experiment; geofence is not a contact sensor.'}
    (Path(output)/'genesis_physical_metrics.json').write_text(json.dumps(result, indent = 2, ensure_ascii = False))
    return result, phys, path


Overwriting genesis_mission.py


In [4]:
%%writefile road_ml.py
"""Road issue classification utilities for the PAI final project.

The Hugging Face road issues data has *issue category* labels, not defect
severity or normal-vs-abnormal labels. The curated risk score below is a
policy assumption (not learned severity, not a calibrated probability).
"""
from __future__ import annotations
import json
from pathlib import Path
import numpy as np

REPO_ID = 'Programmer-RD-AI/road-issues-detection-dataset'
RISK_BY_LABEL = {
    'Pothole Issues': 0.95,
    'Damaged Road issues': 0.75,
    'Broken Road Sign Issues': 0.70,
    'Illegal Parking Issues': 0.55,
    'Mixed Issues': 0.85,
    'Littering Garbage on Public Places Issues': 0.40,
    'Vandalism Issues': 0.35,
}


def label_risk(label: str) -> float:
    if label not in RISK_BY_LABEL:
        raise ValueError(f'Unknown dataset label: {label!r}. Revisit the risk policy mapping.')
    return float(RISK_BY_LABEL[label])


def expected_policy_risk(class_probabilities, class_names):
    """Expected *heuristic* risk: not probability of a severe physical hazard."""
    vals = np.asarray([label_risk(str(n)) for n in class_names], dtype = float)
    p = np.asarray(class_probabilities, dtype = float)
    if p.shape[-1] != len(vals):
        raise ValueError('class dimension mismatch')
    return np.sum(p*vals, axis = -1)


def select_station(current_xy, locations, scores, visited = None, policy = 'adaptive'):
    """Distance vs learned-category-dependent policy comparison.

    Note: straight-line distance is for target priority only, not obstacle-safe
    path length. The physical controller independently uses A*.
    """
    visited = set(visited or ())
    available = [s for s in locations if s not in visited]
    if not available:
        return None
    pos = np.asarray(current_xy, dtype = float)
    dd = {k:float(np.linalg.norm(pos-np.asarray(locations[k]))) for k in available}
    if policy == 'nearest':
        return min(available, key = lambda k:(dd[k], k))
    if policy != 'adaptive':
        raise ValueError('policy should be nearest or adaptive')
    return max(available, key = lambda k:(3.5*float(scores[k]) - 0.30*dd[k], -dd[k]))


def safe_speed_limit(risk, nominal_mps = 0.65):
    """Interpretable, deliberately conservative safety shield (simulation only)."""
    risk = float(np.clip(risk, 0., 1.))
    if risk >= 0.80:
        return min(nominal_mps, 0.20)
    if risk >= 0.60:
        return min(nominal_mps, 0.32)
    if risk >= 0.40:
        return min(nominal_mps, 0.48)
    return float(nominal_mps)


def save_station_manifest(items, path):
    Path(path).write_text(json.dumps(items, ensure_ascii = False, indent = 2), encoding = 'utf8')


class RoadVision:
    def __init__(self, checkpoint_path, device = 'cpu'):
        import torch
        import torchvision
        from torchvision.models import mobilenet_v3_small
        ckpt = torch.load(str(checkpoint_path), map_location = 'cpu', weights_only = True)
        self.class_names = [str(x) for x in ckpt['labels']]
        # Model weights from the training notebook; do not silently use ImageNet labels.
        self.model = mobilenet_v3_small(weights = None, num_classes = len(self.class_names))
        self.model.load_state_dict(ckpt['state_dict'])
        self.device = torch.device(device)
        self.model.to(self.device).eval()
        self.preprocess=torchvision.transforms.Compose([
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean = (.485, .456, .406), std = (.229, .224, .225))
        ])
        self.temperature = float(ckpt.get('temperature', 1.0))
        self.source = ckpt.get('source','unknown')

    def predict(self, img):
        import torch
        from PIL import Image
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        x = self.preprocess(img.convert('RGB')).unsqueeze(0).to(self.device)
        with torch.inference_mode():
            probs = torch.softmax(self.model(x)[0]/self.temperature, dim = 0).cpu().numpy()
        predicted = int(np.argmax(probs))
        return {
            'label':self.class_names[predicted],
            'confidence':float(probs[predicted]),
            'policy_risk':float(expected_policy_risk(probs, self.class_names)),
            'category_probabilities':{n:float(p) for n, p in zip(self.class_names, probs)},
            'risk_is_heuristic':True,
        }


Overwriting road_ml.py


In [5]:
%%writefile genesis_vision_mission.py
"""HuggingFace-trained issue classification -> Genesis mobile inspection rover.

Physical: wheel-joint velocity commands in Genesis, A* waypoints, pose feedback.
Perception: *external virtual roadside cameras* provide held-out HF photographs.
Images are NOT pixels rendered by the Genesis camera. The 3D overhead camera is
for visualization only. This is a coupled benchmark, not sim2real validation.
"""
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from PIL import Image
import json
import math
import numpy as np
import pandas as pd
from road_ml import RoadVision, select_station, safe_speed_limit, label_risk
from genesis_mission import make_scene, xy_yaw, MissionSettings
from pai_patrol import BASE, STATIONS, FloorMap

@dataclass
class VisionSettings(MissionSettings):
    # 20 physics steps/sec, 20 recorded fps: nominal MP4 runtime should be 90 s
    fps:int = 20
    duration_s:float = 90.
    dt:float = .05
    policy:str = 'adaptive'


def load_virtual_camera_feeds(checkpoint, manifest_path):
    model = RoadVision(checkpoint)
    manifest = json.loads(Path(manifest_path).read_text(encoding = 'utf8'))
    feeds = {}
    for entry in manifest:
        station = entry['station']
        if station not in STATIONS:
            raise ValueError('Unrecognized station '+station)
        image_path = Path(manifest_path).parent/entry['image']
        prediction = model.predict(image_path)
        feeds[station] = {'path':str(image_path.resolve()),'truth_category':entry['label'], **prediction}
    if set(feeds) != set(STATIONS):
        raise ValueError('Need one independent HF held-out image per station A..F')
    return feeds


def run_visual_mission(settings, feeds, output, ppo = None):
    import cv2
    output = Path(output)
    output.mkdir(parents = True, exist_ok = True)
    scene, rover, camera, joints = make_scene(settings, output)
    floor = FloorMap()
    visited = set()
    target = None
    route = []
    idx = 0
    risks = {k:feeds[k]['policy_risk'] for k in STATIONS}
    scores = risks if settings.policy == 'adaptive' else {k:0 for k in STATIONS}
    filename=output/f'genesis_visual_{settings.policy}.mp4'
    trace = []
    visits = []
    last_xy = None
    travel = 0.
    first_high_visit = None
    camera.start_recording(save_to_filename = str(filename.resolve()), fps = settings.fps)
    try:
        for step in range(int(settings.duration_s/settings.dt)):
            t = step*settings.dt
            pos, yaw, z = xy_yaw(rover)
            if last_xy is not None:
                travel += math.dist(pos, last_xy)
            last_xy = pos
            if target is None:
                target = select_station(pos, STATIONS, scores, visited, settings.policy)
                if target is not None:
                    route = floor.plan(pos, STATIONS[target])
                    idx = 0
            v = omega = 0.
            local_risk = 0.
            if target is not None:
                distance = math.dist(pos, STATIONS[target])
                local_risk = risks[target]*max(0., 1.-distance/3.)
                if distance<.45:
                    visited.add(target)
                    visits.append({'station':target,'arrival_s':t,'groundtruth_category':feeds[target]['truth_category'],
                                   'predicted_category':feeds[target]['label'],'risk_policy':risks[target]})
                    if label_risk(feeds[target]['truth_category']) >= .8 and first_high_visit is None:
                        first_high_visit = t
                    target = None
                    route = []
                else:
                    if idx >= len(route):
                        route = floor.plan(pos, STATIONS[target])
                        idx = 0
                    if idx<len(route)-1 and math.dist(pos, route[idx])<.27:
                        idx += 1
                    wp = route[idx]
                    angle = math.atan2(wp[1]-pos[1], wp[0]-pos[0])
                    error = (angle-yaw+math.pi)%(2*math.pi)-math.pi
                    omega = float(np.clip(2.0*error, -settings.max_omega, settings.max_omega))
                    max_v = safe_speed_limit(local_risk, settings.max_v)
                    if ppo is not None:
                        obs = np.array([local_risk, 0., min(1., travel/18.), min(1., t/settings.duration_s)], dtype = np.float32)
                        action, _ = ppo.predict(obs, deterministic = True)
                        max_v = min(max_v, float(np.clip(np.asarray(action).reshape(-1)[0], 0, 1))*settings.max_v)
                    v = max_v*max(0., 1.-abs(error)/.8)
            wl = settings.wheel_sign*(v-.5*settings.wheel_track*omega)/settings.wheel_radius
            wr = settings.wheel_sign*(v+.5*settings.wheel_track*omega)/settings.wheel_radius
            rover.control_dofs_velocity(np.asarray([wl, wr]), dofs_idx_local = joints)
            scene.step()
            trace.append({'time_s':t,'x':pos[0],'y':pos[1],'z':z,'target':target or 'NONE',
                          'risk_from_hf':local_risk,'speed_cmd_mps':v,'visited':len(visited)})
    finally:
        try:
            rover.control_dofs_velocity(np.zeros(2), dofs_idx_local = joints)
        except Exception:
            pass
        camera.stop_recording()
    pd.DataFrame(trace).to_csv(output/f'visual_{settings.policy}_physical_trace.csv',index=False)
    pd.DataFrame(visits).to_csv(output/f'visual_{settings.policy}_visits.csv',index=False)
    metrics={'policy':settings.policy,'true_physical_distance_m':travel,
             'visited':len(visited),'first_high_risk_visit_s':first_high_visit,
             'duration_sim_s':settings.duration_s,'video':str(filename),
             'description':'External virtual roadside photo feed -> HF-trained classification -> Genesis real physics motion',
             'not_an_onboard_genesis_camera':True}
    (output/f'visual_{settings.policy}_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf8')
    # A vital assertion that file runtime matches 1-3 min assignment window.
    cap = cv2.VideoCapture(str(filename))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    n = float(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    video_seconds = n/fps if fps>0 else 0
    metrics['observed_mp4_seconds'] = video_seconds
    if not (60 <= video_seconds <= 180):
        raise RuntimeError(f'Physical run succeeded but MP4 duration={video_seconds:.1f}s. Re-encode video for 60-180s; check Genesis recorded FPS.')
    return metrics, pd.DataFrame(trace), filename


Overwriting genesis_vision_mission.py


## 出力フォルダとシークレットを含まない学習済み資産の読み込み

Notebook 04からダウンロードした`road_ml_assets.zip`をアップロードしてください。`HFH_READ`は不要です。

In [6]:
import os
import zipfile
from pathlib import Path
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['PYGLET_HEADLESS'] = 'true'
from google.colab import files

uploaded = files.upload()
# Find any uploaded zip file resembling the expected road assets
uploaded_zip = next((k for k in uploaded.keys() if 'road_ml_assets' in k and k.endswith('.zip')), None)
assert uploaded_zip is not None, 'Upload the file generated by notebook 04'

asset_dir = Path('ml_assets')
asset_dir.mkdir(exist_ok = True)

with zipfile.ZipFile(uploaded_zip) as z:
    for x in z.namelist():
        if x.startswith('/') or '..' in Path(x).parts:
            raise ValueError('Unexpected archive path')
    z.extractall(asset_dir)

from road_ml import RoadVision, label_risk
from genesis_vision_mission import VisionSettings, load_virtual_camera_feeds, run_visual_mission

feeds = load_virtual_camera_feeds(asset_dir / 'road_mobilenetv3.pt', asset_dir / 'station_manifest.json')
for k, v in feeds.items():
    print(k, 'truth:', v['truth_category'], 'pred:', v['label'], 'confidence:', round(v['confidence'], 3), 'policy risk:', round(v['policy_risk'], 3))

Saving road_ml_assets.zip to road_ml_assets.zip
A truth: Broken Road Sign Issues pred: Broken Road Sign Issues confidence: 0.924 policy risk: 0.677
B truth: Damaged Road issues pred: Damaged Road issues confidence: 0.996 policy risk: 0.75
C truth: Illegal Parking Issues pred: Littering Garbage on Public Places Issues confidence: 0.521 policy risk: 0.48
D truth: Littering Garbage on Public Places Issues pred: Littering Garbage on Public Places Issues confidence: 0.977 policy risk: 0.407
E truth: Mixed Issues pred: Mixed Issues confidence: 0.828 policy risk: 0.864
F truth: Pothole Issues pred: Pothole Issues confidence: 0.984 policy risk: 0.948


## Optional PPOモデル（Notebook 05の成果物）

簡易サロゲート上で学習したPPOを速度候補として試します。未アップロードならルールベースの安全速度制限で実行します。

In [7]:
from stable_baselines3 import PPO
USE_PPO = False  # True にするには ppo_speed.zip を Colabのファイル領域に配置
ppo = None
if USE_PPO:
    assert Path('ppo_speed.zip').is_file(),'Upload ppo_speed.zip from notebook 05'
    ppo = PPO.load('ppo_speed.zip', device = 'cpu')
print('PPO used:', bool(ppo))

PPO used: False


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 物理ベースの3Dミッション

90秒×20fpsでの生成を意図したパラメータです。実際のMP4フレーム数を確認し、60〜180秒に収まらなければエラーとします。初回は60秒で描画・動作だけ確認してから本番に進めてください。

In [ ]:
import genesis as gs
import torch
from IPython.display import display, Video
from pathlib import Path
from genesis_vision_mission import VisionSettings, run_visual_mission, load_virtual_camera_feeds

out = Path('outputs')
out.mkdir(exist_ok = True)

try:
    gs.init(backend = gs.gpu if torch.cuda.is_available() else gs.cpu)
except gs.GenesisException:
    print('Genesis is already initialized. Continuing...')

# Ensure feeds is defined in this session
if 'feeds' not in globals():
    asset_dir = Path('ml_assets')
    if (asset_dir / 'road_mobilenetv3.pt').exists() and (asset_dir / 'station_manifest.json').exists():
        feeds = load_virtual_camera_feeds(asset_dir / 'road_mobilenetv3.pt', asset_dir / 'station_manifest.json')
        print("Loaded 'feeds' successfully inside this cell.")
    else:
        raise RuntimeError("Assets not found. Please run cell 025d465a to upload road_ml_assets.zip first.")

# Ensure ppo is defined in this session
if 'ppo' not in globals():
    ppo = None

RUN_3D = True  # 00が未成功ならFalse
RUN_BASELINE = False # True: nearestも独立に実行（Colab時間を多く消費）
if RUN_3D:
    cfg = VisionSettings(duration_s = 90., fps = 20, res = (640, 480), policy = 'adaptive', seed = 42)
    metrics, trace, raw_video = run_visual_mission(cfg, feeds, out, ppo = ppo)
    print(metrics)
    display(trace.tail())
    if RUN_BASELINE:
        cfg2 = VisionSettings(duration_s = 90., fps = 20, res = (640, 480), policy = 'nearest', seed = 42)
        baseline, base_trace, base_video = run_visual_mission(cfg2, feeds, out, ppo = None)
        print('baseline', baseline)
else:
    print('Skipped Genesis run; ML image classification is still testable')

[Genesis] [05:38:14] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [05:38:14] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [05:38:14] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [05:38:14] [INFO] Running on [NVIDIA RTX PRO 6000 Blackwell Server Edition] with backend gs.cuda. Device memory: 94.97 GB.
[Genesis] [05:38:14] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.4, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[Genesis] [05:38:14] [INFO] Scene <88be83d> created.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> plane_15a44fa. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> obstacle_0. idx: 1, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> obstacle_1. idx: 2, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> obstacle_2. idx: 3, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> equipment_A. idx: 4, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> inspect_marker_A. idx: 5, morph: <gs.morphs.Box>, material: <gs.materials.Rigid>.
[Genesis] [05:38:14] [INFO] Adding <gs.engine.entities.RigidEntity> equ

/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node = ast.keyword(
/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/function_def_transformer.py:781: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  return index == 0 and isinstance(stmt, ast.Expr) and isinstance(stmt.value, (ast.Constant, ast.Str))


[Genesis] [05:38:59] [INFO] Building visualizer...


## 録画へ説明オーバーレイを加える

HF画像は**外部仮想点検カメラの静止画**として表示します。Genesisで撮影した画像だと誤認しないよう明記します。

In [ ]:
import cv2
import numpy as np
from PIL import Image
if RUN_3D:
    cap = cv2.VideoCapture(str(raw_video))
    assert cap.isOpened()
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    final = out/'PAI_final_task_HF_Genesis.mp4'
    writer = cv2.VideoWriter(str(final), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    stations = list(feeds)
    thumbnails = {}
    for k, f in feeds.items():
        image = cv2.imread(f['path'])
        thumbnails[k] = cv2.resize(image, (140, 105))
    frames = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        row = trace.iloc[min(len(trace)-1, int(frames*len(trace)/max(1, n)))]
        station = row['target']
        cv2.rectangle(frame, (0, 0), (W, 67), (22, 25, 34), -1)
        cv2.putText(frame,'AI-ADAPTIVE ROAD INSPECTION ROVER', (14, 25), cv2.FONT_HERSHEY_SIMPLEX, .66, (255, 255, 255), 2)
        cv2.putText(frame,f'Elapsed {float(row["time_s"]):.1f}s | Target {station} | Risk {float(row["risk_from_hf"]):.2f} | Speed {float(row["speed_cmd_mps"]):.2f} m/s',
                    (14, 52), cv2.FONT_HERSHEY_SIMPLEX, .44, (255, 225, 145), 1)
        if station in thumbnails and W>360 and H>220:
            frame[75:180, W-152:W-12] = thumbnails[station]
            cv2.putText(frame,'HF VIRTUAL FEED', (W-154, 194), cv2.FONT_HERSHEY_SIMPLEX, .37, (0, 0, 0), 2)
            cv2.putText(frame,'HF VIRTUAL FEED', (W-154, 194), cv2.FONT_HERSHEY_SIMPLEX, .37, (255, 255, 255), 1)
        writer.write(frame)
        frames += 1
    cap.release()
    writer.release()
    seconds = frames/fps
    print('Rendered final MP4:', final, 'runtime_s:', round(seconds, 2))
    assert 60 <= seconds <= 180,'YouTube video must be 1–3 minutes'
    display(Video(str(final), embed = False))

## 提出結果と注意

- 動画は`outputs/PAI_final_task_HF_Genesis.mp4`。
- モデル分類のMacro-F1はNotebook 04の**写真分類タスク**で計算。
- Genesisでの訪問時間・走行距離は本Notebookの**物理結果**で評価。
- リスクスコアは**手作業で決めた重大度指標**であり、深刻度の教師あり推定ではない。
- `PPO`は簡略環境で学習したポリシー。Genesisの直接強化学習ではない。
- 人名、所属、メール、HFH_READ、認証情報を動画・GitHubに含めないでください。

In [ ]:
if RUN_3D:
    import json
    print('Outputs:')
    for f in sorted(out.glob('*')):
        print(' -', f, round(f.stat().st_size/1e6, 2),'MB')
    from google.colab import files
    files.download(str(final))

In [ ]:
import pandas as pd
import json

log_file_path = '/content/app (2).log'
parsed_messages = []

with open(log_file_path, 'r', encoding = 'utf-8', errors = 'ignore') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            data = json.loads(line)
            msg = data.get('msg', '')
            level = data.get('level', 'UNKNOWN')
            time_str = data.get('time', 'UNKNOWN')
            parsed_messages.append({'time': time_str, 'level': level, 'message': msg})
        except Exception:
            parsed_messages.append({'time': None, 'level': 'RAW_LINE', 'message': line})

df_parsed = pd.DataFrame(parsed_messages)

print("=== Summary of parsed messages ===")
print(f"Total log entries: {len(df_parsed)}")
print(f"Level counts: \n{df_parsed['level'].value_counts()}\n")

# Grouping and printing the core warnings/errors
print("=== Filtered warnings & unique messages ===")
unique_msgs = df_parsed['message'].unique()
for i, m in enumerate(unique_msgs[:30]):
    if m.strip():
        print(f"{i+1:2d}. {m.strip()[:120]}")
if len(unique_msgs) > 30:
    print(f"... and {len(unique_msgs) - 30} more unique lines.")

In [ ]:
import json
import re

log_file_path = '/content/app (2).log'
critical_entries = []

# クラッシュや深刻な異常に関連しそうなキーワードを定義
crash_keywords = re.compile(r'crash|fail|error|exception|kill|terminate|fatal|oom|exit', re.IGNORECASE)

with open(log_file_path, 'r', encoding = 'utf-8', errors = 'ignore') as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue

        # JSONとしてパースを試みる
        is_json = False
        msg_text = line
        level = 'UNKNOWN'
        time_str = 'UNKNOWN'

        try:
            data = json.loads(line)
            msg_text = data.get('msg', '')
            level = str(data.get('level', 'UNKNOWN'))
            time_str = data.get('time', 'UNKNOWN')
            is_json = True
        except Exception:
            pass

        # キーワードマッチング
        if crash_keywords.search(msg_text) or crash_keywords.search(str(level)):
            critical_entries.append({
                'line_no': line_num,
                'time': time_str,
                'level': level,
                'message': msg_text,
                'is_json': is_json
            })

print(f"=== Deep Search Results ({len(critical_entries)} suspicious entries found) ===")
if critical_entries:
    for entry in critical_entries:
        print(f"[Line {entry['line_no']}] [Time: {entry['time']}] [Level: {entry['level']}] {entry['message']}")
else:
    print("Jupyterログ内には、直接的なクラッシュ、例外、強制終了を示す明示的なログは見つかりませんでした。")
    print("カーネル自体が突然死（Dead Kernel）している場合、OS側のシステムログ（dmesg等）やメモリ制限（OOM Killer）の可能性があります。")